## Concept Overview

In Logistic Regression, we want to predict the probability that a given input $x$ belongs to a the positive class (1): 

$\hat{y} = P(y = 1 \mid x)$


Since probabilities must stay between $0$ and $1$, we do not simply use a linear function like

$
w^T x + b.
$

Instead, we first define the linear term $z = w^T x + b,$ and then pass it through the **Sigmoid function**, defined as

$$
\sigma(z) = \frac{1}{1 + e^{-z}}.
$$

This S-shaped curve acts as a squashing function:

- When $z$ is a very large positive number, $e^{-z} \to 0$, so  
  $$
  \sigma(z) \to 1.
  $$

- When $z$ is a very large negative number, $e^{-z}$ becomes very large, making the denominator grow significantly, so  
  $$
  \sigma(z) \to 0.
  $$

Thus, the sigmoid function ensures that the output always remains between $0$ and $1$, making it suitable for modeling probabilities.


### Sigmoid
![sigmoid-curve](../images/sigmoid-curve.png)

$\sigma(x) = \frac{1}{1 + e^{-z}}$
- Squashes input into the range (0, 1). 
- Used for binary classification where the output is the probability of a single class being positive
- Suffers from vanishing gradients — when inputs are very positive or very negative, the gradient becomes extremely small, slowing or stopping learning.

In [1]:
import torch

In [2]:
z_values = torch.arange(-4, 5, 2)
z_values

tensor([-4, -2,  0,  2,  4])

In [3]:
torch.sigmoid(z_values)

tensor([0.0180, 0.1192, 0.5000, 0.8808, 0.9820])

## Loss vs. Cost Function

The **Loss function** $\mathcal{L}(\hat{y}, y)$ measures error for a single training example, while the **Cost function** $J(w, b)$ measures the average error across the entire training set of $m$ examples.

**1. The Loss Function (Cross-Entropy)**

To ensure a convex optimization surface, we use:
$$\mathcal{L}(\hat{y}, y) = -(y \log \hat{y} + (1-y) \log(1-\hat{y}))$$

This formula automatically adapts based on the ground truth $y$:

- If $y = 1$: 
$\mathcal{L}(\hat{y}, y) = -\log \hat{y}$
  - Goal: $\hat{y} \rightarrow 1$ (Loss $\rightarrow 0$).
- If $y = 0$: $\mathcal{L}(\hat{y}, y) = -\log(1-\hat{y})$ 
  - Goal: $\hat{y} \rightarrow 0$ (Loss $\rightarrow 0$). 

**2. The Cost Function**

The cost is the arithmetic mean of all individual losses:$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(\hat{y}^{(i)}, y^{(i)})$$Full expanded form:$$J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log \hat{y}^{(i)} + (1-y^{(i)}) \log(1-\hat{y}^{(i)}) \right]$$

**Why do we multiply the loss function by $-1$?**
- Because the logarithm of any number between 0 and 1 is always negative and since Logistic Regression outputs probabilities, $log (\hat{y})$ will always be negative. Therefore, we multiply the result of the loss function with -1 to transform it into a positive loss that gradient descent can minimize. 

In [6]:
torch.log(torch.tensor(0.9))

tensor(-0.1054)

## Deriving the Gradient for Logistic Regression
Let's define the components:

* **Prediction:** $\hat{y} = \sigma(z)$ (where $\sigma$ is the sigmoid function $\frac{1}{1+e^{-z}}$)
* **Linear Logit:** $z = wx + b$
* **Loss:** $\mathcal{L} = -(y \log \hat{y} + (1-y) \log(1-\hat{y}))$

To find the derivative of the loss with respect to $w$, we chain them together:


$$\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}$$

---

### Step 1: Derivative of Loss w.r.t Prediction ($\frac{\partial \mathcal{L}}{\partial \hat{y}}$)

Using the rules we discussed (treating $y$ as a constant), we differentiate the logs:

1. **Differentiate terms:**

$$\frac{\partial \mathcal{L}}{\partial \hat{y}} = -\left( \frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}} \cdot (-1) \right) = -\left( \frac{y}{\hat{y}} - \frac{1-y}{1-\hat{y}} \right)$$


2. **Find common denominator:**

$$-\left( \frac{y(1-\hat{y}) - \hat{y}(1-y)}{\hat{y}(1-\hat{y})} \right)$$


3. **Expand and Simplify:** The numerator becomes $y - y\hat{y} - \hat{y} + y\hat{y} = y - \hat{y}$.
4. **Finalize:** Multiply by the leading negative to get $\hat{y} - y$.

**Result:** $\frac{\partial \mathcal{L}}{\partial \hat{y}} = \frac{\hat{y}-y}{\hat{y}(1-\hat{y})}$

### Step 2: Derivative of Prediction w.r.t Logit ($\frac{\partial \hat{y}}{\partial z}$)

This is a standard property of the Sigmoid function:


$$\frac{\partial \hat{y}}{\partial z} = \sigma(z)(1-\sigma(z)) = \hat{y}(1-\hat{y})$$

### Step 3: Derivative of Logit w.r.t Weight ($\frac{\partial z}{\partial w}$)

Using the "multiplier" rule for $z = wx + b$, where $x$ is the scaling factor:


$$\frac{\partial z}{\partial w} = x$$

---

### Putting it all together

Now we multiply the three parts:


$$\frac{\partial \mathcal{L}}{\partial w} = \underbrace{\left[ \frac{\hat{y}-y}{\hat{y}(1-\hat{y})} \right]}_{\text{Step 1}} \cdot \underbrace{\left[ \hat{y}(1-\hat{y}) \right]}_{\text{Step 2}} \cdot \underbrace{\left[ x \right]}_{\text{Step 3}}$$

Notice how the denominators in Step 1 perfectly cancel out the terms in Step 2!


$$\frac{\partial \mathcal{L}}{\partial w} = (\hat{y} - y)x$$

For the bias **$b$**, Step 3 becomes $\frac{\partial z}{\partial b} = 1$, so:


$$\frac{\partial \mathcal{L}}{\partial b} = \hat{y} - y$$